In [1]:
import mediapipe as mp
import cv2
import numpy as np

print("MediaPipe Version:", mp.__version__)
print("OpenCV Version:", cv2.__version__)

MediaPipe Version: 0.10.35
OpenCV Version: 4.10.0


In [2]:
import scipy
print(scipy.__version__)

1.17.1


In [3]:
import cv2

print(cv2.__version__)
print(hasattr(cv2, "VideoCapture"))

4.10.0
True


In [4]:
import os

print(os.path.exists("face_landmarker.task"))

True


In [5]:
import mediapipe as mp

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="face_landmarker.task"),
    running_mode=VisionRunningMode.VIDEO,
    num_faces=1
)

landmarker = FaceLandmarker.create_from_options(options)

print("Face Landmarker Loaded Successfully!")

Face Landmarker Loaded Successfully!


In [10]:
import cv2
import mediapipe as mp
import numpy as np

cap = cv2.VideoCapture(0)

frame_number = 0

while cap.isOpened():

    success, frame = cap.read()

    if not success:
        break

    frame_number += 1

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    result = landmarker.detect_for_video(mp_image, frame_number)

    if result.face_landmarks:
        landmarks = result.face_landmarks[0]

        for lm in landmarks:
            x = int(lm.x * frame.shape[1])
            y = int(lm.y * frame.shape[0])

            cv2.circle(frame, (x, y), 1, (0, 255, 0), -1)

        cv2.putText(
            frame,
            "Face Detected",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

    cv2.imshow("Face Landmarks", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

In [9]:
import mediapipe as mp

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="face_landmarker.task"),
    running_mode=VisionRunningMode.VIDEO,
    num_faces=1
)

landmarker = FaceLandmarker.create_from_options(options)

print("Face Landmarker Reloaded!")

Face Landmarker Reloaded!


In [11]:
!pip install scipy

In [12]:
from scipy.spatial import distance

In [13]:
from scipy.spatial import distance

def eye_aspect_ratio(eye):

    A = distance.euclidean(eye[1], eye[5])
    B = distance.euclidean(eye[2], eye[4])
    C = distance.euclidean(eye[0], eye[3])

    ear = (A + B) / (2.0 * C)

    return ear

print("EAR Function Ready!")

EAR Function Ready!


In [14]:
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

print("Eye Landmark Indices Loaded!")

Eye Landmark Indices Loaded!


In [15]:
import cv2
import mediapipe as mp
import numpy as np
import time
from scipy.spatial import distance

# Eye landmark indices
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

# EAR calculation
def eye_aspect_ratio(eye):
    A = distance.euclidean(eye[1], eye[5])
    B = distance.euclidean(eye[2], eye[4])
    C = distance.euclidean(eye[0], eye[3])

    return (A + B) / (2.0 * C)

# Webcam
cap = cv2.VideoCapture(0)

blink_counter = 0
frames_closed = 0

EAR_THRESHOLD = 0.22
CONSEC_FRAMES = 2

while cap.isOpened():

    success, frame = cap.read()

    if not success:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    timestamp = int(time.time() * 1000)

    result = landmarker.detect_for_video(mp_image, timestamp)

    if result.face_landmarks:

        landmarks = result.face_landmarks[0]

        left_eye = []
        right_eye = []

        h, w = frame.shape[:2]

        for idx in LEFT_EYE:
            lm = landmarks[idx]
            left_eye.append((lm.x * w, lm.y * h))

        for idx in RIGHT_EYE:
            lm = landmarks[idx]
            right_eye.append((lm.x * w, lm.y * h))

        leftEAR = eye_aspect_ratio(left_eye)
        rightEAR = eye_aspect_ratio(right_eye)

        ear = (leftEAR + rightEAR) / 2.0

        if ear < EAR_THRESHOLD:
            frames_closed += 1
        else:
            if frames_closed >= CONSEC_FRAMES:
                blink_counter += 1
            frames_closed = 0

        cv2.putText(
            frame,
            f"Blinks: {blink_counter}",
            (20,40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0,255,0),
            2
        )

        cv2.putText(
            frame,
            f"EAR: {ear:.2f}",
            (20,80),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (255,0,0),
            2
        )

    cv2.imshow("Blink Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()